# Notebook 3 – Hyperparameters

**Dataset:** Online Retail transactions (`data.csv`)

**Task:** Predict whether an order is from the **United Kingdom** or not, using `Quantity`, `UnitPrice`, `TotalPrice`.


## Setup: Load & Split Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
df = pd.read_csv('data.csv', encoding='latin1')
df = df.dropna(subset=['CustomerID'])
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)].sample(3000, random_state=42)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
X = df[['Quantity', 'UnitPrice', 'TotalPrice']]
y = (df['Country'] == 'United Kingdom').astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 1. Parameters vs Hyperparameters
- **Parameters** are learned *from the data* during training (e.g., the coefficients in Logistic Regression, or the splits in a Decision Tree).
- **Hyperparameters** are set *before* training by us — they control *how* the model learns, and are not learned from data.

In [2]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(C=1.0)  
model.fit(X_train, y_train)
print("Learned parameters (coefficients):", model.coef_)  
print("Hyperparameter used:", model.C)                     

Learned parameters (coefficients): [[-0.00224306 -0.01597315  0.00026923]]
Hyperparameter used: 1.0


## 2. Why Hyperparameters Matter
The same algorithm can perform very differently depending on its hyperparameters — too simple underfits, too complex overfits. Picking good hyperparameters is often the difference between a mediocre and a great model.

In [4]:
from sklearn.tree import DecisionTreeClassifier
for depth in [1, 4, None]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    print(f"max_depth={depth}: Test accuracy={accuracy_score(y_test, m.predict(X_test)):.3f}")

max_depth=1: Test accuracy=0.897
max_depth=4: Test accuracy=0.893
max_depth=None: Test accuracy=0.868


## 3. Model Configuration
Hyperparameters together define a model's **configuration**. Changing them doesn't require new data — it just changes how the same algorithm behaves on the same data.

In [5]:
config_A = DecisionTreeClassifier(max_depth=3, criterion='gini')
config_B = DecisionTreeClassifier(max_depth=3, criterion='entropy')
print("Config A:", config_A.get_params()['criterion'])
print("Config B:", config_B.get_params()['criterion'])

Config A: gini
Config B: entropy


## 4. Common Hyperparameters
Below we go through the most important hyperparameters for 5 popular algorithms, one at a time.

In [6]:
print("See sections below: Decision Tree, Random Forest, KNN, SVM, Logistic Regression")

See sections below: Decision Tree, Random Forest, KNN, SVM, Logistic Regression


### Decision Tree — `max_depth`
Controls how deep the tree can grow. Small `max_depth` → simple tree (underfitting risk). Large/unlimited → complex tree (overfitting risk).

In [7]:
for d in [2, 5, None]:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    print(f"max_depth={d}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

max_depth=2: Test acc=0.897
max_depth=5: Test acc=0.892
max_depth=None: Test acc=0.868


### Decision Tree — `min_samples_split`
The minimum number of samples a node must have before it's allowed to split further. Higher values make the tree simpler and less likely to overfit.

In [8]:
for mss in [2, 50, 200]:
    m = DecisionTreeClassifier(min_samples_split=mss, random_state=42).fit(X_train, y_train)
    print(f"min_samples_split={mss}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

min_samples_split=2: Test acc=0.868
min_samples_split=50: Test acc=0.872
min_samples_split=200: Test acc=0.890


### Decision Tree — `min_samples_leaf`
The minimum number of samples allowed in a final leaf node. Higher values prevent the tree from creating tiny, overly specific leaves (reduces overfitting).

In [9]:
for msl in [1, 20, 100]:
    m = DecisionTreeClassifier(min_samples_leaf=msl, random_state=42).fit(X_train, y_train)
    print(f"min_samples_leaf={msl}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

min_samples_leaf=1: Test acc=0.868
min_samples_leaf=20: Test acc=0.897
min_samples_leaf=100: Test acc=0.897


### Decision Tree — `criterion`
The formula used to decide the best way to split a node. `'gini'` measures impurity; `'entropy'` measures information gain. Both usually give similar results.

In [10]:
for c in ['gini', 'entropy']:
    m = DecisionTreeClassifier(criterion=c, max_depth=4, random_state=42).fit(X_train, y_train)
    print(f"criterion={c}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

criterion=gini: Test acc=0.893
criterion=entropy: Test acc=0.895


### Random Forest — `n_estimators`
The number of individual trees in the forest. More trees usually improve stability and accuracy, but with diminishing returns and slower training.

In [11]:
from sklearn.ensemble import RandomForestClassifier
for n in [10, 50, 150]:
    m = RandomForestClassifier(n_estimators=n, random_state=42).fit(X_train, y_train)
    print(f"n_estimators={n}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

n_estimators=10: Test acc=0.880
n_estimators=50: Test acc=0.875
n_estimators=150: Test acc=0.870


### Random Forest — `max_depth`
Same idea as in a single Decision Tree, but applied to every tree in the forest — controls how deep each tree can grow.

In [12]:
for d in [3, 8, None]:
    m = RandomForestClassifier(max_depth=d, n_estimators=50, random_state=42).fit(X_train, y_train)
    print(f"max_depth={d}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

max_depth=3: Test acc=0.898
max_depth=8: Test acc=0.895
max_depth=None: Test acc=0.875


### Random Forest — `max_features`
How many features each tree considers when looking for the best split. Lower values add more randomness/diversity between trees (can reduce overfitting).

In [13]:
for mf in [1, 2, 'sqrt']:
    m = RandomForestClassifier(max_features=mf, n_estimators=50, random_state=42).fit(X_train, y_train)
    print(f"max_features={mf}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

max_features=1: Test acc=0.875
max_features=2: Test acc=0.877
max_features=sqrt: Test acc=0.875


### Random Forest — `min_samples_split`
Same meaning as in a Decision Tree — minimum samples needed to split a node — applied across all trees in the forest.

In [14]:
for mss in [2, 20, 100]:
    m = RandomForestClassifier(min_samples_split=mss, n_estimators=50, random_state=42).fit(X_train, y_train)
    print(f"min_samples_split={mss}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

min_samples_split=2: Test acc=0.875
min_samples_split=20: Test acc=0.893
min_samples_split=100: Test acc=0.897


### KNN — `n_neighbors`
How many nearby points "vote" on a prediction. Small values → sensitive to noise (overfitting risk). Large values → smoother, more general decision boundary (underfitting risk if too large).

In [15]:
from sklearn.neighbors import KNeighborsClassifier
for k in [1, 5, 50]:
    m = KNeighborsClassifier(n_neighbors=k).fit(X_train, y_train)
    print(f"n_neighbors={k}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

n_neighbors=1: Test acc=0.823
n_neighbors=5: Test acc=0.882
n_neighbors=50: Test acc=0.897


### KNN — `weights`
How much influence each neighbor has. `'uniform'` treats all neighbors equally; `'distance'` gives closer neighbors more influence than farther ones.

In [16]:
for w in ['uniform', 'distance']:
    m = KNeighborsClassifier(n_neighbors=10, weights=w).fit(X_train, y_train)
    print(f"weights={w}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

weights=uniform: Test acc=0.892
weights=distance: Test acc=0.875


### KNN — `metric`
How "distance" between points is measured. `'euclidean'` is straight-line distance; `'manhattan'` sums absolute differences along each axis.

In [17]:
for metric in ['euclidean', 'manhattan']:
    m = KNeighborsClassifier(n_neighbors=10, metric=metric).fit(X_train, y_train)
    print(f"metric={metric}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

metric=euclidean: Test acc=0.892
metric=manhattan: Test acc=0.893


### SVM — `C`
Controls the trade-off between a wider margin and correctly classifying every training point. Small `C` → wider margin, more tolerant of errors (simpler). Large `C` → tries hard to classify everything correctly (can overfit).

In [20]:
from sklearn.svm import SVC
for c in [0.1, 1, 10]:
    m = SVC(C=c).fit(X_train, y_train)
    print(f"C={c}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

C=0.1: Test acc=0.897
C=1: Test acc=0.897
C=10: Test acc=0.893


### SVM — `kernel`
The function used to separate classes. `'linear'` draws a straight boundary; `'rbf'` can draw curved, more flexible boundaries for non-linear data.

In [21]:
for k in ['linear', 'rbf']:
    m = SVC(kernel=k).fit(X_train, y_train)
    print(f"kernel={k}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

kernel=linear: Test acc=0.897
kernel=rbf: Test acc=0.897


### SVM — `gamma`
(Used with the `'rbf'` kernel.) Controls how far the influence of a single training point reaches. Low `gamma` → far-reaching influence (smoother boundary). High `gamma` → very local influence (can overfit).

In [22]:
for g in [0.001, 0.1, 1]:
    m = SVC(kernel='rbf', gamma=g).fit(X_train, y_train)
    print(f"gamma={g}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

gamma=0.001: Test acc=0.897
gamma=0.1: Test acc=0.893
gamma=1: Test acc=0.890


### Logistic Regression — `C`
The inverse of regularization strength. Small `C` → stronger regularization (simpler model). Large `C` → weaker regularization (fits training data more closely).

In [23]:
for c in [0.01, 1, 100]:
    m = LogisticRegression(C=c, max_iter=1000).fit(X_train, y_train)
    print(f"C={c}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

C=0.01: Test acc=0.897
C=1: Test acc=0.897
C=100: Test acc=0.897


### Logistic Regression — `penalty`
Which type of regularization to apply: `'l1'` (can zero out coefficients), `'l2'` (shrinks them smoothly), or `'elasticnet'` (a mix of both).

In [24]:
m1 = LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000).fit(X_train, y_train)
m2 = LogisticRegression(penalty='l2', max_iter=1000).fit(X_train, y_train)
print("L1 Test acc:", accuracy_score(y_test, m1.predict(X_test)))
print("L2 Test acc:", accuracy_score(y_test, m2.predict(X_test)))

L1 Test acc: 0.8966666666666666
L2 Test acc: 0.8966666666666666


C:\Users\hemak\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\hemak\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\hemak\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default va

### Logistic Regression — `solver`
The optimization algorithm used to find the best coefficients. Different solvers support different penalties and scale differently with data size (e.g., `'liblinear'` for small data, `'saga'` for large data with L1/Elastic Net).

In [25]:
for solver in ['liblinear', 'lbfgs']:
    m = LogisticRegression(solver=solver, max_iter=1000).fit(X_train, y_train)
    print(f"solver={solver}: Test acc={accuracy_score(y_test, m.predict(X_test)):.3f}")

solver=liblinear: Test acc=0.897
solver=lbfgs: Test acc=0.897
